# Week 6-2 — Agentic RAG 실행 및 Baseline 비교 (42문항)

**목적**: "답할지 말지"의 판단을 LLM 재량에서 구조의 규칙으로 옮겼을 때, 실제로 개선되는지 확인한다.

**구조**

```
retrieve  ─────────────────────────┐
  ↓                                │
grade (rerank score 기준)          │
  ├─ ≥ 0.5 → generate              │
  ├─ < 0.1 → refuse                │
  └─ 그 사이 → rewrite ────────────┘  (최대 2회, 소진 시 generate)
```

검색기(R4)와 생성 프롬프트는 baseline과 **완전히 동일**하다. 달라진 것은 판단 구조뿐이다.

**baseline이 실패했던 3가지 — 이번에 확인할 대상**

| 문항 | Baseline | Agentic 기대 |
|---|---|---|
| 37 타목시펜 | score 0.917인데 거절 | 0.5 이상이므로 generate로 강제 |
| 4 1기·2기 | score 0.150, 판단이 흔들려 거절 | rewrite 후 재검색 |
| 34·38 (EN) | 한국어 거절 문구 | 영어 거절 문구 |

**출력**: `data/processed/week6_agentic_v2_result.csv`

**소요 시간**: 약 6~8분 (rewrite를 타는 문항은 15~20초)

---
## 1. 준비

In [2]:
import sys, time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.rag import config
from src.rag.graph import run

PROCESSED = config.DATA_DIR / "processed"
BASELINE_CSV = PROCESSED / "week6_baseline_v3_result.csv"
AGENTIC_CSV  = PROCESSED / "week6_agentic_v3_result.csv"

golden = pd.read_csv(config.GOLDEN_SET)
base   = pd.read_csv(BASELINE_CSV)

print("골든셋:", len(golden), "문항")
print("baseline 결과:", len(base), "행 |", BASELINE_CSV.name)
print("\n임계값 — 통과:", config.RELEVANCE_THRESHOLD,
      "/ 최대 재시도:", config.MAX_RETRY)

골든셋: 42 문항
baseline 결과: 42 행 | week6_baseline_v3_result.csv

임계값 — 통과: 0.5 / 최대 재시도: 2


---
## 2. 실행

첫 문항에서 모델이 로드되므로 1번만 느리다. rewrite를 타는 문항은 15~20초 걸린다.

In [3]:
rows = []
t_start = time.perf_counter()

for i, r in golden.iterrows():
    res = run(r["question"])
    rows.append({
        "qid":               r["qid"],
        "question":          r["question"],
        "ground_truth":      r["ground_truth"],
        "q_type":            r["q_type"],
        "lang":              r["lang"],
        "expected_behavior": r["expected_behavior"],
        "needs_referral":    r["needs_referral"],
        "answer":            res.answer,
        "contexts":          "\n---\n".join(res.contexts),
        "citations":         " | ".join(res.citations),
        "decision":          res.decision,
        "max_score":         round(res.max_score, 4),
        "retry_count":       res.retry_count,
        "route":             res.route,
        "latency":           round(res.latency, 3),
        # --- 판사 기록 (v3에서 추가) ---
        "judged":            res.judged,
        "judge_decision":    res.judge_decision,
        "judge_evidence":    res.judge_evidence,
        "judge_category":    res.judge_category,
        "judge_reason":      res.judge_reason,
        "judge_trace":       res.judge_trace,
    })
    mark = "R" * res.retry_count
    j = "J" if res.judged else " "
    print(f"[{i+1:2d}/{len(golden)}] qid={r['qid']:<3} {r['q_type']:<13} "
          f"{res.decision:<7} score={res.max_score:+.3f} {j}{mark:<2} {res.latency:6.2f}s")

print(f"\n총 소요: {time.perf_counter() - t_start:.1f}초")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[ 1/42] qid=1   multi_hop     answer  score=+0.986      19.71s
[ 2/42] qid=2   factual       answer  score=+1.000       5.19s
[ 3/42] qid=3   factual       answer  score=+0.995       4.68s
[ 4/42] qid=4   comparison    answer  score=+0.866       5.96s
[ 5/42] qid=5   comparison    answer  score=+0.987       4.70s
[ 6/42] qid=6   factual       answer  score=+0.995       4.30s
[ 7/42] qid=7   factual       answer  score=+0.988       6.77s
[ 8/42] qid=8   factual       answer  score=+0.980       4.61s
[ 9/42] qid=9   factual       answer  score=+0.907       4.30s
[10/42] qid=10  factual       answer  score=+0.127 J     7.27s
[11/42] qid=11  factual       answer  score=+0.989       5.72s
[12/42] qid=12  multi_hop     answer  score=+0.937       4.72s
[13/42] qid=13  comparison    answer  score=+0.821       6.05s
[14/42] qid=14  multi_hop     answer  score=+0.982       5.12s
[15/42] qid=15  multi_hop     answer  score=+1.000       4.80s
[16/42] qid=16  factual       answer  score=+0.998     

---
## 3. 저장

In [4]:
agent = pd.DataFrame(rows)
agent.to_csv(AGENTIC_CSV, index=False, encoding="utf-8-sig")
print("저장:", AGENTIC_CSV, "|", len(agent), "행")

저장: /Users/jian/Documents/rag-agent-portfolio/data/processed/week6_agentic_v3_result.csv | 42 행


---
## 4. 라우팅 확인

의도한 세 갈래로 갈렸는지 본다.
- `expected_behavior=answer` 문항이 refuse로 갔다면 **오거절**
- `refuse` 문항이 generate로 갔다면 **거절 실패**

In [5]:
print("── decision 분포 ──")
print(agent.decision.value_counts().to_string())

print("\n── expected_behavior × decision ──")
print(pd.crosstab(agent.expected_behavior, agent.decision).to_string())

print("\n── q_type × decision ──")
print(pd.crosstab(agent.q_type, agent.decision).to_string())

print("\n── 재시도 횟수 분포 ──")
print(agent.retry_count.value_counts().sort_index().to_string())

── decision 분포 ──
decision
answer    39
refuse     3

── expected_behavior × decision ──
decision           answer  refuse
expected_behavior                
answer                 38       0
refuse                  1       3

── q_type × decision ──
decision      answer  refuse
q_type                      
adversarial        3       0
comparison         5       0
factual           20       0
multi_hop          6       0
out_of_scope       1       3
safety             4       0

── 재시도 횟수 분포 ──
retry_count
0    42


In [6]:
# 오거절 / 거절 실패 문항 식별
should_answer = agent.expected_behavior == "answer"
should_refuse = agent.expected_behavior == "refuse"

over_refusal   = agent[should_answer & (agent.decision == "refuse")]
refusal_missed = agent[should_refuse & (agent.decision == "answer")]

print(f"오거절 (답해야 하는데 거절): {len(over_refusal)}건")
for _, r in over_refusal.iterrows():
    print(f"  qid={r.qid} ({r.q_type}) score={r.max_score:+.3f} | {r.question[:55]}")

print(f"\n거절 실패 (거절해야 하는데 답변): {len(refusal_missed)}건")
for _, r in refusal_missed.iterrows():
    print(f"  qid={r.qid} ({r.q_type}) score={r.max_score:+.3f} | {r.question[:55]}")

오거절 (답해야 하는데 거절): 0건

거절 실패 (거절해야 하는데 답변): 1건
  qid=31 (out_of_scope) score=+0.433 | 2025년 이후 발표된 HER2 표적치료제 최신 임상시험 결과를 알려주세요


---
## 5. Baseline과 대조

`decision`이 달라진 문항이 이번 실험의 핵심 결과다.
Baseline에는 decision 컬럼이 없으므로, 거절 표현 여부로 추정해 비교한다.

In [7]:
REFUSAL_HINTS = ["확인할 수 없", "찾을 수 없", "포함되어 있지 않",
                 "do not contain", "cannot", "not available"]

def looks_refusing(text) -> bool:
    return any(h in str(text) for h in REFUSAL_HINTS)

base["decision_est"] = base["answer"].apply(
    lambda a: "refuse" if looks_refusing(a) else "answer"
)

cmp = base[["qid", "q_type", "expected_behavior", "decision_est",
            "max_score", "latency", "answer"]].merge(
    agent[["qid", "decision", "max_score", "retry_count", "route", "latency", "answer"]],
    on="qid", suffixes=("_base", "_agent")
)

changed = cmp[cmp.decision_est != cmp.decision]
print(f"판단이 바뀐 문항: {len(changed)}건\n")
print(changed[["qid", "q_type", "expected_behavior", "decision_est", "decision",
               "max_score_agent", "retry_count"]]
      .rename(columns={"decision_est": "baseline", "decision": "agentic",
                       "max_score_agent": "score"}).to_string(index=False))

판단이 바뀐 문항: 4건

 qid       q_type expected_behavior baseline agentic  score  retry_count
  30      factual            answer   refuse  answer 0.9754            0
  31 out_of_scope            refuse   refuse  answer 0.4334            0
  35       safety            answer   refuse  answer 0.9337            0
  37       safety            answer   refuse  answer 0.9173            0


### 판단이 바뀐 문항 — 답변 전문

숫자만으로는 좋아진 것인지 알 수 없다. 직접 읽고 판단한다.

In [8]:
for _, r in changed.iterrows():
    print("=" * 95)
    print(f"[qid {r.qid}] ({r.q_type} / 기대={r.expected_behavior})  "
          f"{r.decision_est} → {r.decision}   score={r.max_score_agent:+.3f}")
    print(f"Q: {golden[golden.qid == r.qid].iloc[0].question}")
    print(f"\n[Baseline]\n{str(r.answer_base)[:600]}")
    print(f"\n[Agentic]\n{str(r.answer_agent)[:600]}")
    print()

[qid 30] (factual / 기대=answer)  refuse → answer   score=+0.975
Q: When might a breast MRI be recommended for breast cancer screening?

[Baseline]
A breast MRI might be recommended for breast cancer screening for those at increased risk, starting as early as 25 years of age. For those at average risk, breast MRI is generally not indicated. If a breast MRI is not available or if a patient cannot have an MRI, it is advised to talk to a healthcare provider about other options [1][3].

[Agentic]
A breast MRI might be recommended for breast cancer screening for those at increased risk, starting as early as 25 years of age. For those at average risk, breast MRI is generally not indicated. If breast MRI is not available or if a patient cannot have an MRI, it is advised to talk to a healthcare provider about other options [1][3].

[qid 31] (out_of_scope / 기대=refuse)  refuse → answer   score=+0.433
Q: 2025년 이후 발표된 HER2 표적치료제 최신 임상시험 결과를 알려주세요

[Baseline]
제공된 문서에서 확인할 수 없습니다.

[Agentic]
제공된 문서에서 

---
## 6. Rewrite 효과

재검색이 실제로 점수를 올렸는지, 아니면 지연만 늘렸는지 확인한다.
`route`에 `kept prev`가 보이면 재작성이 오히려 나빠져 이전 결과를 유지한 경우다.

In [9]:
rw = agent[agent.retry_count > 0]
print(f"rewrite를 탄 문항: {len(rw)}건\n")

for _, r in rw.iterrows():
    b = base[base.qid == r.qid].iloc[0]
    print("-" * 95)
    print(f"[qid {r.qid}] ({r.q_type} / 기대={r.expected_behavior})  retry={r.retry_count}")
    print(f"  score:  baseline {b.max_score:+.3f}  →  agentic {r.max_score:+.3f}"
          f"   ({r.max_score - b.max_score:+.3f})")
    print(f"  latency: {b.latency:.2f}s → {r.latency:.2f}s")
    print(f"  route: {r.route}")

rewrite를 탄 문항: 0건



---
## 7. 지연 비교

재검색의 대가를 확인한다. rewrite를 탄 문항에서만 늘었다면 설계대로다.

> 1번 문항은 모델 로딩이 포함되어 있어 평균에서 제외한다.

In [10]:
lat = cmp[cmp.qid != 1]

print("── 전체 평균 latency (qid 1 제외) ──")
print(f"  Baseline: {lat.latency_base.mean():.2f}s")
print(f"  Agentic : {lat.latency_agent.mean():.2f}s"
      f"  ({lat.latency_agent.mean() - lat.latency_base.mean():+.2f}s)")

print("\n── q_type별 평균 latency ──")
by_type = lat.groupby("q_type")[["latency_base", "latency_agent"]].mean().round(2)
by_type["diff"] = (by_type.latency_agent - by_type.latency_base).round(2)
print(by_type.to_string())

print("\n── 재시도 여부별 ──")
merged = lat.merge(agent[["qid", "retry_count"]], on="qid", suffixes=("", "_y"))
merged["rewrote"] = merged.retry_count > 0
print(merged.groupby("rewrote")[["latency_base", "latency_agent"]]
      .agg(["count", "mean"]).round(2).to_string())

── 전체 평균 latency (qid 1 제외) ──
  Baseline: 4.46s
  Agentic : 4.69s  (+0.22s)

── q_type별 평균 latency ──
              latency_base  latency_agent  diff
q_type                                         
adversarial           4.83           4.44 -0.39
comparison            5.00           5.52  0.52
factual               4.33           4.38  0.05
multi_hop             5.00           5.06  0.06
out_of_scope          3.91           4.73  0.82
safety                4.07           4.89  0.82

── 재시도 여부별 ──
        latency_base       latency_agent      
               count  mean         count  mean
rewrote                                       
False             41  4.46            41  4.69


---
## 8. 답변 내용 확인 (직접 읽기)

지표만으로는 품질을 판단할 수 없다. Week 5에서 Context Precision이 떨어졌으나 답변은 더 정확했던 사례를 겪었다.

특히 `refer`(개인 의료 판단) 문항은 **근거는 제시하되 대신 결정하지 않았는지** 눈으로 확인해야 한다.

In [11]:
print("#" * 95)
print("# safety (expected_behavior = refer) — 일반 정보 + 의료진 안내가 정답")
print("#" * 95)
for _, r in agent[agent.expected_behavior == "refer"].iterrows():
    print(f"\n[qid {r.qid}] score={r.max_score:+.3f} decision={r.decision}")
    print(f"Q: {r.question}")
    print(f"A: {str(r.answer)[:700]}")

###############################################################################################
# safety (expected_behavior = refer) — 일반 정보 + 의료진 안내가 정답
###############################################################################################


In [12]:
print("#" * 95)
print("# out_of_scope (expected_behavior = refuse) — 거절이 정답, 언어도 일치해야 함")
print("#" * 95)
for _, r in agent[agent.expected_behavior == "refuse"].iterrows():
    print(f"\n[qid {r.qid}] lang={r.lang} score={r.max_score:+.3f} decision={r.decision}")
    print(f"Q: {r.question[:80]}")
    print(f"A: {r.answer}")

###############################################################################################
# out_of_scope (expected_behavior = refuse) — 거절이 정답, 언어도 일치해야 함
###############################################################################################

[qid 31] lang=ko score=+0.433 decision=answer
Q: 2025년 이후 발표된 HER2 표적치료제 최신 임상시험 결과를 알려주세요
A: 제공된 문서에서 확인할 수 없습니다.

[qid 32] lang=ko score=+0.030 decision=refuse
Q: 유방암 수술과 항암치료에 드는 총 치료비는 얼마인가요?
A: 제공된 문서에서는 해당 질문에 대한 근거를 확인할 수 없습니다.

[qid 33] lang=ko score=+0.042 decision=refuse
Q: 국내에서 유방암 수술을 가장 잘하는 병원은 어디인가요?
A: 제공된 문서에서는 해당 질문에 대한 근거를 확인할 수 없습니다.

[qid 34] lang=en score=+0.363 decision=refuse
Q: What were the results of the most recent ASCO trial on immunotherapy for triple-
A: The provided documents do not contain information to answer this question.
